# Fine-Tuning Qwen2.5:7b Multi-Task & Morphological Induction (Sangiang v3)
**Proyek:** Sangihe Translation Engine & Conversational AI (Sangiang v3)
**Inisiator:** Yayasan Cipta Mandala Dwipa | **AI & Translation:** Divisi DigiLab CMD
**Framework:** Unsloth (Fast QLoRA) + Hugging Face TRL
**Target Export:** GGUF (Q4_K_M) untuk Ollama lokal

### Inovasi Sangiang v3:
1. **Disambiguasi Pronomina Kuat**: Menyelesaikan kekeliruan homograf `iạ` (aku/saya) vs bahasa Indonesia `ia` (dia), serta klitik `-u` (-mu) vs `-ne` (-nya).
2. **Morphological Induction CoT (Ide Pengguna)**: Melatih model dengan 10.000 pasang penalaran dekonstruksi kata dasar, imbuhan, sisipan (-um-, -in-), dan hukum pergeseran makna leksikon Steller.
3. **Response-Only Masking**: Menggunakan `train_on_responses_only` agar bobot neural terkonsentrasi 100% pada ketepatan terjemahan dan penalaran morfologis.

### 1. Instalasi Dependensi (Unsloth & TRL)

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes


### 2. Memuat Base Model Qwen2.5:7b-Instruct & Konfigurasi LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512 # Optimal untuk pasangan kalimat, leksikon, dan penalaran morfologi
dtype = None # Auto detection (Float16 untuk T4 GPU)
load_in_4bit = True # 4bit QLoRA menghemat 70% VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Pasang Adapter LoRA pada layer proyeksi atensi dan FFN
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model Qwen2.5:7b siap untuk fine-tuning Sangiang v3!")


### 3. Unggah Dataset Sangiang v3
Unggah file `sangiang_v3_colab_data.zip` (4 MB) dari komputer lokal Anda.

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("train.jsonl"):
    print("Silakan unggah sangiang_v3_colab_data.zip:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".zip"):
            with zipfile.ZipFile(filename, "r") as z:
                z.extractall(".")
            print("Ekstraksi berhasil: train.jsonl dan val.jsonl siap digunakan!")
else:
    print("train.jsonl dan val.jsonl sudah tersedia.")


### 4. Format Dataset dengan Chat Template Qwen2.5

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convs]
    return {"text": texts}

raw_train = load_dataset("json", data_files="train.jsonl", split="train")
raw_val = load_dataset("json", data_files="val.jsonl", split="train")

train_dataset = raw_train.map(formatting_prompts_func, batched=True)
val_dataset = raw_val.map(formatting_prompts_func, batched=True)

print(f"Total data latih: {len(train_dataset)} pasang instruksi")
print(f"Total data validasi: {len(val_dataset)} pasang instruksi")
print("Contoh data latih:")
print(train_dataset[0]["text"][:350])


### 5. Memulai Pelatihan SFT (Supervised Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Evaluasi dibatasi ke 100 sampel agar cepat dan mencegah bottleneck I/O
eval_subsample = val_dataset.select(range(min(100, len(val_dataset))))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_subsample,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=30,
        max_steps=1000, # Durasi: ~20-25 menit di GPU T4
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=500,
    ),
)

print("Memulai pelatihan Sangiang v3 (Estimasi: 20-25 menit)...\n")
trainer_stats = trainer.train()


### 6. Uji Inferensi Hasil Fine-Tuning Sangiang v3

In [ ]:
FastLanguageModel.for_inference(model)

def test_prompt(user_text, system_prompt=None):
    if not system_prompt:
        system_prompt = "Nama Anda adalah Sangiang. Anda adalah kecerdasan buatan, filolog, dan penerjemah ahli bahasa Sangihe ke bahasa Indonesia formal. Ingat: 'iạ' adalah kata ganti orang pertama (aku/saya), bukan 'ia' (dia)."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return reply.strip()

print("=== TEST 1: Pronomina Orang Pertama 'iạ' (Saya/Aku) ===")
print("User: Terjemahkan teks bahasa Sangihe berikut ke bahasa Indonesia formal:\nIạ kai mạngangěndung.")
print("Sangiang:", test_prompt("Terjemahkan teks bahasa Sangihe berikut ke bahasa Indonesia formal:\nIạ kai mạngangěndung."))
print()

print("=== TEST 2: Klitik Posesif '-u' (Sarangmu) & Partikel '-ko' ===")
print("User: Terjemahkan teks bahasa Sangihe berikut ke bahasa Indonesia formal:\nMahi iạko měbaesẹ̌ su waes-u.")
print("Sangiang:", test_prompt("Terjemahkan teks bahasa Sangihe berikut ke bahasa Indonesia formal:\nMahi iạko měbaesẹ̌ su waes-u."))
print()

print("=== TEST 3: Penalaran Morfologis & Pergeseran Makna (Ide Pengguna) ===")
prompt_morph = "Lakukan analisis morfologis pada kata 'měbaesẹ̌' dalam bahasa Sangihe. Dekonstruksi kata dasar, imbuhan, dan pergeseran maknanya!"
print("User:", prompt_morph)
print("Sangiang:", test_prompt(prompt_morph))
print()

print("=== TEST 4: Sapaan Percakapan Santun ===")
print("User: Tabea! Kere habarẹ̌?")
print("Sangiang:", test_prompt("Tabea! Kere habarẹ̌?"))


### 7. Ekspor Model GGUF (Q4_K_M) ke Google Drive
Mengonversi bobot model yang telah dilatih ke format GGUF Q4_K_M untuk diimpor ke Ollama.

In [ ]:
import os, glob, shutil
from google.colab import drive, files
local_gguf_dir = "/content/sangiang_v3_gguf"
os.makedirs(local_gguf_dir, exist_ok=True)
print("1. Mengonversi bobot model Sangiang v3 ke format GGUF (Q4_K_M)...")
model.save_pretrained_gguf(local_gguf_dir, tokenizer, quantization_method="q4_k_m")
gguf_files = glob.glob(f"{local_gguf_dir}*/*.gguf")
if gguf_files:
    gguf_path = gguf_files[0]
    size_gb = os.path.getsize(gguf_path) / (1024**3)
    print(f"\n✅ GGUF Berhasil Dibuat di Disk Lokal:")
    print(f"   Path: {gguf_path}")
    print(f"   Ukuran: {size_gb:.2f} GB\n")
    # Opsi A: Salin ke Google Drive
    try:
        print("2. Menghubungkan Google Drive...")
        drive.mount('/content/drive')
        drive_dest = '/content/drive/MyDrive/Sangihe_Translation_Models'
        os.makedirs(drive_dest, exist_ok=True)
        dest_file = os.path.join(drive_dest, "sangiang_v3_q4_k_m.gguf")
        print(f"   Menyalin ke {dest_file}...")
        shutil.copyfile(gguf_path, dest_file)
        drive.flush_and_unmount()
        print("   ✅ Selesai tersimpan di Google Drive sebagai sangiang_v3_q4_k_m.gguf!")
    except Exception as e:
        print(f"   ⚠️ Gagal menyalin ke Google Drive: {e}")
    # Opsi B: Unduh langsung lewat browser
    print("\n3. OPSI UNDUH LANGSUNG:")
    print("   Jalankan perintah berikut bila ingin mengunduh langsung ke laptop:")
    print(f"   files.download('{gguf_path}')")
else:
    print("❌ File GGUF belum ditemukan. Periksa keluaran log konversi di atas.")


### 8. Sinkronisasi Final Google Drive

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('Sinkronisasi selesai! Berkas sangiang_v3_q4_k_m.gguf telah siap di Google Drive.')
